# RAER — stage 00: pre-flight audit

Run this **before** any labeling or training. It answers the questions that decide
whether the dataset is usable at all: what actually ships, whether `P###` is a real
subject id, how the classes are distributed, whether an official split is included,
and how badly a naive split would leak subjects.

Two stages, cheapest first:

* **A — manifest only.** Reads Kaggle's file listing (names + sizes). Answers the
  clip, identity and leakage questions in about a minute, with **no download**.
* **B — full copy.** ~100 GB. Only worth it once stage A looks right.

See `DATA_PROVENANCE.md` for the licensing position. The Kaggle copy is an interim
source while official access is pending; nothing derived from it gets published.

## 1. Setup

In [ ]:
# Both repos: RAER-external for this task, LLMSTU-pipeline for the shared labeler.
RAER_REPO   = "vaelkokach/RAER-external"
LLMSTU_REPO = "vaelkokach/LLMSTU-pipeline"

import os, sys, subprocess
try:
    from google.colab import userdata
    def _secret(n):
        try: return userdata.get(n)
        except Exception: return None
except Exception:
    def _secret(n): return os.environ.get(n)

gh = _secret("GITHUB_TOKEN")
for repo in (RAER_REPO, LLMSTU_REPO):
    name = repo.split("/")[-1]
    if not os.path.isdir(f"/content/{name}"):
        auth = f"{gh}@" if gh else ""
        subprocess.run(["git","clone","-q",f"https://{auth}github.com/{repo}.git",
                        f"/content/{name}"], check=True)

sys.path[:0] = ["/content/RAER-external", "/content/LLMSTU-pipeline"]
os.chdir("/content/RAER-external")
print("cwd:", os.getcwd())

In [ ]:
!pip install -q kaggle pandas

In [ ]:
# Derived artifacts live on Drive so they survive the runtime dying.
# The MEDIA stays on local disk — it is large, slow over Drive, and must not be shared.
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE = Path("/content/drive/MyDrive/RAER")       # derived artifacts only
LOCAL = Path("/content/raer")                      # media, never synced to Drive
for d in (DRIVE, LOCAL): d.mkdir(parents=True, exist_ok=True)
print("drive:", DRIVE, "| local:", LOCAL)
!df -h /content | tail -1

In [ ]:
# Kaggle credentials — add KAGGLE_USERNAME and KAGGLE_KEY to Colab Secrets (key icon).
os.environ["KAGGLE_USERNAME"] = _secret("KAGGLE_USERNAME") or ""
os.environ["KAGGLE_KEY"]      = _secret("KAGGLE_KEY") or ""
assert os.environ["KAGGLE_KEY"], "Set KAGGLE_USERNAME / KAGGLE_KEY in Colab Secrets"

DATASET = "bearmn/raer-video-emotion-dataset"

## 2. Stage A — audit the manifest (no download)

If the clip count is far off 2,649, or the identity verdict is not what we expect,
stop here rather than spending 100 GB and an hour finding out.

In [ ]:
from raer import kaggle_listing, audit

listing = kaggle_listing.fetch(DATASET)
kaggle_listing.save(listing, DRIVE / "audit" / "listing.json")
print(f"{len(listing):,} files, {sum(s for _, s in listing)/1e9:.1f} GB")

r = audit.run(listing=listing, out_dir=DRIVE / "audit", source=f"kaggle:{DATASET}")

In [ ]:
print("clips        ", r["clips"]["n_clips"], "vs paper", r["clips"]["paper_n_clips"],
      f"({r['clips']['delta_vs_paper']:+d})")
print("frames/clip  ", r["clips"]["frames_per_clip"])
print("identity     ", r["identity"]["verdict"])
print("split key    ", r["identity"]["safe_split_key"])
print("leakage      ", r["leakage"]["verdict"])

In [ ]:
from IPython.display import HTML
HTML((DRIVE / "audit" / "audit_report.html").read_text())

## 3. Fetch just the small metadata files

The label mapping and any split lists are a few MB. Pull only those, then re-run the
audit with them so the distribution and split checks can complete.

In [ ]:
small = [p for p, s in listing
         if Path(p).suffix.lower() in {".json", ".txt", ".csv", ".tsv"} and s < 50_000_000]
print("
".join(f"{p}" for p in small))

meta_dir = LOCAL / "meta"
kaggle_listing.download_files(DATASET, small, meta_dir)
!ls -la {meta_dir}

In [ ]:
# Unzip anything Kaggle wrapped, then look at what we actually got.
!cd {meta_dir} && for z in *.zip; do [ -e "$z" ] && unzip -o -q "$z" && rm "$z"; done; ls -la

labels = audit.find_labels(meta_dir)
splits = audit.find_splits(meta_dir)
print(f"labels recovered: {len(labels):,}")
print("splits:", {k: len(v) for k, v in splits.items()})

dist = audit.class_distribution(labels)
print(dist["status"])
import pandas as pd
pd.DataFrame(dist["rows"])

In [ ]:
print(audit.split_check(splits))

## 4. Stage B — full copy (only if stage A is clean)

~100 GB. Check free disk first: Colab gives ~235 GB with ~47 GB already used, so this
fits, but not alongside a second copy. Downloading unzipped avoids needing 2x the space.

In [ ]:
!df -h /content | tail -1
# uncomment to run — this takes a while
# !kaggle datasets download -d {DATASET} -p {LOCAL} --unzip

In [ ]:
# Full audit over the downloaded copy
# r_full = audit.run(root=LOCAL / "RAER", out_dir=DRIVE / "audit_full")
# HTML((DRIVE / "audit_full" / "audit_report.html").read_text())

## 5. Gate

Do not proceed to stage 01 (frame sampling) until every line below is resolved and
written into `DATA_PROVENANCE.md`:

- [ ] clip count reconciles with 2,649 — the mirror has 2,669 mp4, explain the 20
- [ ] identity verdict is known, and the safe split key is recorded
- [ ] class distribution matches the published 65.23 / 19.89 / 7.40 / 6.08 / 1.40
- [ ] official split either recovered, or we commit to building our own subject-disjoint one
- [ ] the two `.json` files are understood — whose captions are they?
- [ ] official access request resolved, or the interim source is documented as provisional